# 📘 11.3 Layer Normalization — Revision Notes

> **Core idea:** Layer Normalization normalizes the features **within each individual sample**, rather than using statistics across a mini-batch.

![Image](https://images.openai.com/static-rsc-4/hCWgCGzIAD1ct_UV4MMFBiD7BCFxSSG-xRt78RdeTYJSIZf4uYNMROrG3mNAqe5hHakJQC3Mf_AEoRATS1Ou6zUdOGCFTO3-3Pq_rwf9-mrh6qFlL_ZfY5f1auwNuC9ZOPGdGNEb_F0fqsTN4a8sFJziHkZjUP4wh6SkX6Pjwap1fIHWf-fAiMYvCuJeayV1?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/zUlI7CUnLh6PU4ZAZoAZ1bn9TvBcDET6iO2-oDdw2mVttbF620ySAH68NHhy8jyDo_2KdWfXXSAe5Zcl2EKfkrpe8ePz7r9Suo1jA_UMXa5E7kDx8Msylmmw1JiHW7Z-Jjm8n2hGCvvkZnfyPjmlGqSi021znD6EpMsT4uD_ISZlBE-BfN0oEoINvVP2v-PJ?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/c0QKUccutQlmMrTohpEHlj5EffXeSgdIIQFmaVvUUHcQmObFzlQY5zv2L67RjVrvuSCioYuE3w66rhOasQAfNhDSrXij6v4tqb2TRyNuOC7PjG3mWm3vH303_011mYNvBJ7Q3rjunbyOLd883ikkNaDpZ0KpiRVHAUe-Zt1zHxXwZgKgHzwmK3gFe6CozpPD?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/wyR01ALUfdWqnLOngUk61bV-tJnZHRNELe-0GjfLtl1LZGhBVIDyQn-VHo08XiD-f0bIlsvn3JgLS41fhEq5VP1ga1Ga4dKq8jm3RnLuXuaOnKktHD3WaPfrUmodR7Kwfb9k9X8zRilthRpo3-Je0ZrmHd8JoTKLO7KASec65uDru9moedMsuIr4ywQ3JDL5?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/P17-OBVcyWDqqbYzQqDTk7cWMPq0l-JakVO0PISxLNvvlDSm7Et9VS4kWMfaCqiLMFaAWFvEkqSKgEsUp72f6k9ibVd70_c_Ys87CKvmFlrfaGNgXyIlidAFsFEvcCst0bqCXn-zx-tZkL0vCwxd05XGqrH5rkB_APGxKPXZyqXR8T5wVxsOpd0dWY3bbeZm?purpose=fullsize)

---

## 1. What is Layer Normalization?

Layer Normalization (LayerNorm) standardizes activations **independently for each sample**.

For one sample:

```text
Features
   ↓
[x₁  x₂  x₃  x₄  x₅]
   ↓
Calculate mean & variance
   ↓
Normalize
   ↓
Scale (γ)
   ↓
Shift (β)
   ↓
Output
```

The important point is:

> **LayerNorm does not need other samples in the batch to calculate its statistics.**

The original LayerNorm paper introduced it as an alternative to BatchNorm whose statistics are computed from the activations of a single training case rather than across a mini-batch. ([arXiv][1])

---

# 2. Simple Example

Suppose our batch is:

$$
X=
\begin{bmatrix}
2&4&6&8\\
1&3&5&7\\
4&6&8&10
\end{bmatrix}
$$

There are:

* 3 samples
* 4 features per sample

```text
             Features
          f₁  f₂  f₃  f₄

Sample 1   2   4   6   8
Sample 2   1   3   5   7
Sample 3   4   6   8  10
```

For **LayerNorm**, we process each row independently.

### Sample 1

$$
[2,4,6,8]
$$

Mean:

$$
\mu=\frac{2+4+6+8}{4}=5
$$

Variance:

$$
\sigma^2=
\frac{(2-5)^2+(4-5)^2+(6-5)^2+(8-5)^2}{4}
=5
$$

Then normalize each feature.

Sample 2 and Sample 3 get their **own** mean and variance.

---

# 3. LayerNorm Equation 🔥

For a sample with $d$ features:

### Mean

$$
\mu=\frac{1}{d}\sum_{i=1}^{d}x_i
$$

### Variance

$$
\sigma^2=
\frac{1}{d}
\sum_{i=1}^{d}(x_i-\mu)^2
$$

### Normalize

$$
\hat{x}_i=
\frac{x_i-\mu}
{\sqrt{\sigma^2+\epsilon}}
$$

### Scale and shift

$$
\boxed{
y_i=\gamma_i\hat{x}_i+\beta_i
}
$$

So the complete operation is:

$$
\boxed{
y_i=
\gamma_i
\frac{x_i-\mu}
{\sqrt{\sigma^2+\epsilon}}
+\beta_i
}
$$

The PyTorch implementation follows this same structure and calculates statistics over the last dimensions specified by `normalized_shape`. ([PyTorch Docs][2])

---

# 4. The MOST Important Difference: What Dimension? 🔥🔥🔥

This is the part you absolutely need to remember.

Suppose:

```text
        f1   f2   f3   f4
S1      ●    ●    ●    ●
S2      ●    ●    ●    ●
S3      ●    ●    ●    ●
```

## BatchNorm

For each feature, look **down the batch**:

```text
f1
●  ← S1
●  ← S2
●  ← S3
↑
statistics across batch
```

Then separately:

```text
f2
●
●
●
↑
statistics across batch
```

So:

$$
\boxed{\text{BatchNorm → across the batch}}
$$

---

## LayerNorm

For each sample, look **across its features**:

```text
S1 → ● ● ● ●
      └─────┘
      statistics

S2 → ● ● ● ●
      └─────┘
      statistics

S3 → ● ● ● ●
      └─────┘
      statistics
```

So:

$$
\boxed{\text{LayerNorm → across features}}
$$

![Image](https://images.openai.com/static-rsc-4/hCWgCGzIAD1ct_UV4MMFBiD7BCFxSSG-xRt78RdeTYJSIZf4uYNMROrG3mNAqe5hHakJQC3Mf_AEoRATS1Ou6zUdOGCFTO3-3Pq_rwf9-mrh6qFlL_ZfY5f1auwNuC9ZOPGdGNEb_F0fqsTN4a8sFJziHkZjUP4wh6SkX6Pjwap1fIHWf-fAiMYvCuJeayV1?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/wyR01ALUfdWqnLOngUk61bV-tJnZHRNELe-0GjfLtl1LZGhBVIDyQn-VHo08XiD-f0bIlsvn3JgLS41fhEq5VP1ga1Ga4dKq8jm3RnLuXuaOnKktHD3WaPfrUmodR7Kwfb9k9X8zRilthRpo3-Je0ZrmHd8JoTKLO7KASec65uDru9moedMsuIr4ywQ3JDL5?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/mBrx13QUm9oOkXZKSO-W8Y5UHCQp52x1bsureiCpeftP43lo1OTj3DFoE5x6T_7pAmhIgNhmd1aT2PZ7c-mjmEDHPLdNwNxUVMpeuJlbSQZ7lCPD6wKQ1Ytz5QiDH-qyWAKwKKZfubaWCovVVUdgeZZSyskqx10dY8cNdt6fG1-HLUD4-nOQCzaLEK_JHGAg?purpose=fullsize)

---

# 5. BatchNorm vs LayerNorm 🔥🔥🔥

| Property                  | BatchNorm                       | LayerNorm                      |
| ------------------------- | ------------------------------- | ------------------------------ |
| Normalizes                | Features using batch statistics | Features within each sample    |
| Statistics depend on      | Other samples in batch          | Only current sample            |
| Batch-size dependent      | Yes                             | No                             |
| Training statistics       | Current batch                   | Current sample                 |
| Inference statistics      | Running mean/variance           | Current sample                 |
| Running statistics needed | Yes                             | No                             |
| Common use                | CNNs / conventional networks    | Transformers / sequence models |
| Learnable scale/shift     | Yes                             | Yes                            |

### Memory trick 🧠

> **BatchNorm = Batch matters**

> **LayerNorm = Individual sample**

---

# 6. Why Doesn't LayerNorm Need Running Statistics?

This is an important consequence of the previous section.

### BatchNorm

During training:

$$
\mu_B,\sigma_B^2
$$

come from the current batch.

During inference, it uses stored running statistics.

```text
Training
   ↓
Batch statistics
   ↓
Running statistics
   ↓
Inference
```

### LayerNorm

For every input sample:

$$
\mu,\sigma^2
$$

are calculated directly from that sample.

```text
Sample
  ↓
Its own mean
  ↓
Its own variance
  ↓
Normalize
```

Therefore, LayerNorm uses input statistics in both training and evaluation. PyTorch's documentation explicitly notes that LayerNorm uses statistics computed from the input in both modes. ([PyTorch Docs][2])

---

# 7. Why is LayerNorm Useful?

A major advantage is that the normalization doesn't depend on the composition or size of the mini-batch.

Imagine:

```text
Batch A:
Sample 1
Sample 2
Sample 3
Sample 4
```

BatchNorm's statistics depend on those samples.

Change the batch:

```text
Batch B:
Sample 5
Sample 6
Sample 7
Sample 8
```

and the statistics can change.

LayerNorm instead does:

```text
Sample 1 → normalize its own features
Sample 2 → normalize its own features
Sample 3 → normalize its own features
...
```

So the operation is much more naturally **sample-independent**.

The original LayerNorm paper specifically highlighted BatchNorm's dependence on mini-batch size and its less straightforward application to recurrent networks as motivations for LayerNorm. ([arXiv][1])

---

# 8. Why LayerNorm is Important in Transformers 🔥

LayerNorm is widely used in Transformer architectures.

A simplified Transformer block looks like:

```text
Input
  │
  ├───────────────┐
  ↓               │
Self-Attention    │
  ↓               │
LayerNorm /       │
Residual          │
  ↓               │
Feed Forward      │
  ↓               │
LayerNorm /       │
Residual          │
  ↓
Output
```

Modern Transformer implementations can use different placements of LayerNorm, such as **pre-norm** or **post-norm**. PyTorch's `TransformerEncoderLayer`, for example, exposes `norm_first` to control whether LayerNorm is applied before or after the attention/feed-forward sublayers. ([PyTorch Docs][3])

![Image](https://images.openai.com/static-rsc-4/uGvM9zpbjqT89U2d_g7K0-dQjKA08P_BjBi0bhtFm2SYnjyhd5zgjqTAB-Kns-tRlGNUL_rLCWt23yfkw3HR7B2gNniPVJBqHRQqrpIpxCWZySE5ZYpViDAltvcV82N9-0_slL3UP9I9pFst5q7NBFu8-Xbcq7xAVAX2wzzKqAJVtL-42lBrRHOatxncXImw?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/hCpdS135wveiknfIVeH2FwAmwX1eoYCD9eXM5VSSEEhqS9KiEURWvbgVGegrZZkmzRZAr4pozVaxAHYOOUNAjxQXKZ9yHuPb_VgVW-QNTtDdg3WL-UA7tzoRxZcU8Pc1z5Kt95700i5ZXXtW2_8aifNa-hZ2agsb9zaDsVOj8SPJimmQ_MB2U1ri_RMBI3OU?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/0PVON_J0pQZbNxTnmPSKtkSQnVSBKsLJPzyYRel4t4xVsIEOGiTcnd07lS4UTbMVIR6nZf5LZDOxGEUycFtDd3wNd6HJjH7BbJ9rAJyLdQafvAj4V4WRtLKrA8rOgqaPkq8R1H2tc7P6mCD6jpjCPIyQWnH60tb3aZC5QHCJg_OECpfn4re0dp3TTNd0ub3k?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/iAW1kZ22CoeW5xxEMJ2BQUtEvTPT3W6r-X4wcoAAB57eNeeQXMnjcqh67b-fNEUzLbkBoekOrm_IK2orNqlmptiyvWkky09q0Sla2SOSXvwoRV7XBuDhfCuqsreV_0aGPQmHRPie6Cqiw66jQLCdMHlCuiDqvOsrSvdDZwUfGtBhYu9d91n8wK91XF-gbNVJ?purpose=fullsize)

![Image](https://images.openai.com/static-rsc-4/Epl9PFQ3P-IrmcYSiIUt-FQ3dlKBsIagak8Hnl55iWT_QXK2oG2_M9UJfI7-BVcWBHnEPoUyZtfCp5vWNm29veiCnlVawCWOTg6wsAVUmazTZOsgni-_TsnGA5SlgEAbOy1UaMsKtf3BO3HVxAmPopD0Pu7u7cZZf7Yxc5YJK8kOp0b9-AORjyx3Uf00byCo?purpose=fullsize)

You don't need to study **pre-norm vs post-norm** yet unless it appears later in your Transformer block.

For now:

> **Transformer → LayerNorm is a very important normalization technique to know.**

---

# 9. $\gamma$ and $\beta$

Just like BatchNorm, LayerNorm has learnable parameters.

$$
y_i=\gamma_i\hat{x}_i+\beta_i
$$

### $\gamma$

Controls scaling:

$$
\gamma>1
\rightarrow
\text{increase scale}
$$

$$
0<\gamma<1
\rightarrow
\text{decrease scale}
$$

### $\beta$

Controls shifting:

$$
\beta>0
\rightarrow
\text{shift upward}
$$

$$
\beta<0
\rightarrow
\text{shift downward}
$$

So:

> **$\gamma$ = learnable scale**

> **$\beta$ = learnable shift**

In PyTorch, these are learnable per-element affine parameters when `elementwise_affine=True`. ([PyTorch Docs][2])

---

# 10. LayerNorm vs BatchNorm — Visual Memory

```text
              BATCH NORMALIZATION

        f1    f2    f3    f4
S1      ●     ●     ●     ●
        ↑     ↑     ↑     ↑
S2      ●     ●     ●     ●
        ↑     ↑     ↑     ↑
S3      ●     ●     ●     ●
        ↑     ↑     ↑     ↑
        └─────┴─────┴─────┘
          statistics use
          multiple samples
```

```text
              LAYER NORMALIZATION

S1      ● ─── ● ─── ● ─── ●
        └─────────────────┘
          statistics for S1

S2      ● ─── ● ─── ● ─── ●
        └─────────────────┘
          statistics for S2

S3      ● ─── ● ─── ● ─── ●
        └─────────────────┘
          statistics for S3
```

The distinction is also illustrated in educational material comparing the axes used by BatchNorm and LayerNorm.

---

# 11. Important Terminology

### Batch dimension

Represents different samples:

$$
N
$$

Example:

```text
32 samples
```

### Feature dimension

Represents features of each sample:

$$
D
$$

Example:

```text
768 features
```

For a Transformer input:

$$
X\in\mathbb{R}^{N\times L\times D}
$$

where:

* $N$ = batch size
* $L$ = sequence length
* $D$ = embedding/feature dimension

LayerNorm commonly normalizes over the feature dimension $D$ for Transformer representations.

For example, PyTorch:

```python
nn.LayerNorm(768)
```

normalizes the last dimension when the input's last dimension is 768. ([PyTorch Docs][2])

---

# 12. Common Mistake ❌

Don't say:

> "LayerNorm normalizes the entire layer."

That's ambiguous.

A better statement is:

> **LayerNorm calculates mean and variance across the specified feature dimensions for each individual sample.**

This is important because `normalized_shape` determines exactly which trailing dimensions PyTorch normalizes. ([PyTorch Docs][2])

---

# 13. LayerNorm vs BatchNorm — Interview Answer 🎯

If an interviewer asks:

> **"What is the difference between BatchNorm and LayerNorm?"**

A strong concise answer is:

> **BatchNorm computes normalization statistics across the batch for each feature, so its behavior depends on the batch and it uses running statistics during inference. LayerNorm computes statistics across the features of each individual sample, so it doesn't depend on other samples in the batch and uses the input's statistics during both training and evaluation. LayerNorm is therefore widely used in Transformer architectures.**

---

# 🧠 Final Cheat Sheet

```text
                    NORMALIZATION
                         │
              ┌──────────┴──────────┐
              ↓                     ↓
         BatchNorm              LayerNorm
              │                     │
       Across batch            Across features
              │                     │
      Batch-dependent          Sample-independent
              │                     │
      Running statistics       No running statistics
       for inference               needed
              │                     │
       Common in CNNs          Common in Transformers
```

### Equations

**LayerNorm mean:**

$$
\mu=\frac{1}{d}\sum_{i=1}^{d}x_i
$$

**Variance:**

$$
\sigma^2=
\frac{1}{d}
\sum_{i=1}^{d}(x_i-\mu)^2
$$

**Normalization:**

$$
\hat{x}_i=
\frac{x_i-\mu}
{\sqrt{\sigma^2+\epsilon}}
$$

**Scale + shift:**

$$
\boxed{y_i=\gamma_i\hat{x}_i+\beta_i}
$$

### 🔥 One sentence to remember

> **BatchNorm asks: "How does this feature behave across the batch?"**

> **LayerNorm asks: "How do this sample's features behave relative to each other?"**

### Useful references

* [Original Layer Normalization paper — Ba, Kiros & Hinton](https://arxiv.org/abs/1607.06450?utm_source=chatgpt.com)
* [PyTorch LayerNorm documentation](https://docs.pytorch.org/docs/stable/generated/torch.nn.modules.normalization.LayerNorm.html?utm_source=chatgpt.com)


